# 01 - Bulk-Phasentabelle aus HDF berechnen

Dieses Notebook zeigt die Bulk-Rechnung. Es lädt `CuGabulk_oxide.hdf`, wertet
die energieabhängigen Terme auf einem Raster aus Temperatur `T` und
`log10(pH2O/pH2)` aus und erzeugt eine stabile Bulk-Phasentabelle.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

# Compatibility shim for HDF files written in a different NumPy/PyTables stack.
# Keep this cell at the top before calling pd.read_hdf.
try:
    import numpy.core as npc
    sys.modules.setdefault("numpy._core", npc)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
except Exception as exc:
    print("NumPy compatibility shim skipped:", exc)

DATA_ROOT = Path(r"/Users/dk2994/Desktop/Uni/Journal/Thesis/Notebooks/Surface Alloys")
PROJECT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI")
OUTPUT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI/notebooks/phase_diagram_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

HDF_FILES = {
    "bulk": DATA_ROOT / "CuGabulk.hdf",
    "bulk_oxide": DATA_ROOT / "CuGabulk_oxide.hdf",
    "surface_100": DATA_ROOT / "CuGasurf_100.hdf",
    "surface_110": DATA_ROOT / "CuGasurf_110.hdf",
    "surface_111": DATA_ROOT / "CuGasurf_111.hdf",
    "surface_211": DATA_ROOT / "CuGasurf_211.hdf",
}

def read_onepiece_hdf(path, key="df"):
    """Read a OnePiece-exported pandas HDF table.

    OnePiece stores simulation records in a pandas DataFrame. The HDF files in
    this tutorial are read with pd.read_hdf(filename, key="df").
    """
    path = Path(path)
    frame = pd.read_hdf(path, key=key)
    frame.attrs["source_hdf"] = str(path)
    return frame

def formula_counts(formula):
    if not isinstance(formula, str):
        return {}
    counts = {}
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts

In [ ]:
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

bulk = read_onepiece_hdf(HDF_FILES["bulk_oxide"])
print(bulk.shape)
bulk.head()

## Thermodynamisches Raster

In [ ]:
CONSTANTS = {
    "H2_E": -6.737,
    "H2O_E": -12.062,
    "H2_S": 1.5044e-3,
    "H2O_S": 2.2198e-3,
    "kb": 8.617333262145e-5,
}

T_values = np.linspace(300, 1100, 161)
log10_ratio_values = np.linspace(-16, 4, 201)
TT, YY_log10 = np.meshgrid(T_values, log10_ratio_values)
LOGR_NATURAL = np.log(10.0) * YY_log10

muO_grid = (
    CONSTANTS["H2O_E"]
    - CONSTANTS["H2_E"]
    - TT * (CONSTANTS["H2O_S"] - CONSTANTS["H2_S"])
    + CONSTANTS["kb"] * TT * LOGR_NATURAL
)

float(muO_grid.min()), float(muO_grid.max())

## Energieausdrücke aus der OnePiece-Tabelle evaluieren

In [ ]:
candidate_columns = [
    "formation_energy_per_atom",
    "formation_energy",
    "form_E_per_atom",
    "form_G_per_atom",
]
[c for c in candidate_columns if c in bulk.columns]

In [ ]:
T, kb, pH2O, pH2, H2O_E, H2_E, H2O_S, H2_S = sp.symbols(
    "T kb pH2O pH2 H2O_E H2_E H2O_S H2_S"
)

def expression_to_grid(expr):
    """Convert a symbolic/string/numeric energy expression to a 2D grid."""
    if pd.isna(expr):
        return np.full_like(TT, np.nan, dtype=float)
    if isinstance(expr, (int, float, np.number)):
        return np.full_like(TT, float(expr), dtype=float)
    sym_expr = sp.sympify(str(expr))
    subs = {
        H2O_E: CONSTANTS["H2O_E"],
        H2_E: CONSTANTS["H2_E"],
        H2O_S: CONSTANTS["H2O_S"],
        H2_S: CONSTANTS["H2_S"],
        kb: CONSTANTS["kb"],
        pH2: 1,
        pH2O: sp.exp(sp.Symbol("logR")),
    }
    logR = sp.Symbol("logR")
    sym_expr = sym_expr.subs(subs)
    func = sp.lambdify((T, logR), sym_expr, "numpy")
    values = func(TT, LOGR_NATURAL)
    return np.asarray(values, dtype=float) + np.zeros_like(TT)

energy_column = next(c for c in candidate_columns if c in bulk.columns)
energy_surfaces = np.stack([expression_to_grid(v) for v in bulk[energy_column]])
stable_index = np.nanargmin(energy_surfaces, axis=0)
stable_energy = np.nanmin(energy_surfaces, axis=0)
energy_surfaces.shape, stable_index.shape

## Stabile Bulk-Phasen zusammenfassen

In [ ]:
def phase_label(row):
    if "Ga_percent" in row and pd.notna(row["Ga_percent"]):
        return f"{row['Ga_percent']:.1f}% Ga"
    counts = formula_counts(row.get("Formula"))
    total = sum(counts.values())
    ga_percent = 100 * counts.get("Ga", 0) / total if total else np.nan
    return f"{ga_percent:.1f}% Ga" if pd.notna(ga_percent) else str(row.get("Name"))

records = []
for phase_id in np.unique(stable_index):
    mask = stable_index == phase_id
    row = bulk.iloc[int(phase_id)]
    counts = formula_counts(row.get("Formula"))
    total = sum(counts.values())
    records.append({
        "phase_id": int(phase_id),
        "Name": row.get("Name", f"phase {phase_id}"),
        "Formula": row.get("Formula", ""),
        "phase_label": phase_label(row),
        "Cu_atoms": counts.get("Cu", np.nan),
        "Ga_atoms": counts.get("Ga", np.nan),
        "Ga_percent": 100 * counts.get("Ga", 0) / total if total else np.nan,
        "stable_grid_fraction": float(mask.mean()),
        "stable_percent": float(100 * mask.mean()),
        "T_min_K": float(TT[mask].min()),
        "T_max_K": float(TT[mask].max()),
        "log10_ratio_min": float(YY_log10[mask].min()),
        "log10_ratio_max": float(YY_log10[mask].max()),
        "min_energy": float(stable_energy[mask].min()),
        "unit": "eV/atom",
        "panel": "Bulk oxide-derived",
    })

bulk_summary = pd.DataFrame(records).sort_values("stable_grid_fraction", ascending=False)
bulk_summary

In [ ]:
bulk_summary.to_csv(OUTPUT_ROOT / "tutorial_bulk_transition_summary.csv", index=False)

## Kontrollplot

In [ ]:
stable_phase_ids = np.unique(stable_index)
remap = {old: new for new, old in enumerate(stable_phase_ids)}
stable_compact = np.vectorize(remap.get)(stable_index)

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
cmap = plt.get_cmap("tab20", len(stable_phase_ids))
mesh = ax.pcolormesh(TT, YY_log10, stable_compact, cmap=cmap, shading="auto")
ax.set_xlabel("Temperature T [K]")
ax.set_ylabel("log10(pH2O/pH2)")
ax.set_title("Bulk stable phase fields")
cbar = fig.colorbar(mesh, ax=ax, ticks=np.arange(len(stable_phase_ids)) + 0.5)
cbar.ax.set_yticklabels([str(bulk.iloc[i].get("Name", i)) for i in stable_phase_ids])
fig.savefig(OUTPUT_ROOT / "tutorial_bulk_phase_fields.png", dpi=180)
plt.show()